In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
import sys, os
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, "config.py")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
if _root not in sys.path: sys.path.insert(0, _root)

import importlib, json, numpy as np, pandas as pd
import config
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from cprics.io_data import load_peak_amplitudes
from cprics.continuous import build_all_combined
from cprics.tracking import track_islands
from cprics.features import build_feature_table
from cprics.classify import assign_archetype
from cprics.metrics import site_eval_table, scorecard

P = importlib.import_module("experiments.exp01_continuous_9R").PARAMS
pk  = load_peak_amplitudes(config.PEAK_AMP_DIR, P["channel"])
isl = track_islands(build_all_combined(pk, wavelength=P["wavelength"],
                                       candidate_floor=P["candidate_floor"]),
                    max_center_gap=P["tracking_gap_m"])

feats = build_feature_table(isl, min_obs=3)
feats = feats.merge(isl.groupby("site_id")["hard_amplitude"].max().ge(30).rename("critical"), on="site_id")
feats = feats.merge(isl.groupby("site_id")["amplitude"].max().rename("cont_max"), on="site_id")
with open(config.MODELS_DIR / "exp01_continuous_9R_archetypes.json") as fh:
    feats["archetype"] = assign_archetype(feats, json.load(fh))
arch = dict(zip(feats.site_id, feats.archetype))

y = feats["critical"].astype(int).values
print("sites:", len(y), "critical:", int(y.sum()))

sites: 2263 critical: 22


In [8]:
# model + honest leave-one-site-out escalation probabilities
def loo_scorecard(cols, label):
    pipe = make_pipeline(StandardScaler(),
                         LogisticRegression(class_weight="balanced", max_iter=2000))
    oof = cross_val_predict(pipe, feats[cols].values, y, cv=LeaveOneOut(),
                            method="predict_proba")[:, 1]
    T = site_eval_table(isl, dict(zip(feats.site_id, oof)), arch, P["concern_mm"], P["critical_mm"])
    m = scorecard(T)
    print(f"{label:28} AUPRC {m['auprc']:.3f}  P@20 {m['precision_at_k'][20]:.3f}  "
          f"R@20 {m['recall_at_k'][20]:.3f}  P@50 {m['precision_at_k'][50]:.3f}")
    return m

FULL  = ["current_amp","mean_amp","max_amp","volatility","slope_overall","slope_current","n_resets"]
BEHAV = ["volatility","slope_overall","slope_current","n_resets"]

print("baseline exp01 (risk score):   AUPRC 0.589  P@20 0.650  R@20 0.591  P@50 0.340")
m_full  = loo_scorecard(FULL,  "exp02 supervised (full):")
m_behav = loo_scorecard(BEHAV, "exp02 behaviour-only:")

baseline exp01 (risk score):   AUPRC 0.589  P@20 0.650  R@20 0.591  P@50 0.340
exp02 supervised (full):     AUPRC 0.783  P@20 0.700  R@20 0.636  P@50 0.420
exp02 behaviour-only:        AUPRC 0.586  P@20 0.600  R@20 0.545  P@50 0.300


In [9]:
# which features drive escalation (standardised logistic coefficients, fit on all data)
pipe = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000)).fit(feats[FULL].values, y)
print(pd.Series(pipe.named_steps["logisticregression"].coef_[0], index=FULL)
      .sort_values(key=abs, ascending=False).round(2).to_string())

max_amp          2.70
mean_amp         0.97
current_amp      0.75
n_resets         0.51
slope_overall    0.42
volatility       0.27
slope_current   -0.19


In [10]:
# where does the AUPRC jump come from? nested feature sets
for cols, name in [
    (["current_amp"],                           "current only"),
    (["current_amp","slope_current"],           "current + growth (~risk score)"),
    (["current_amp","slope_current","max_amp"], "+ historical peak"),
    (FULL,                                      "full"),
]:
    loo_scorecard(cols, name)

current only                 AUPRC 0.682  P@20 0.650  R@20 0.591  P@50 0.380
current + growth (~risk score) AUPRC 0.684  P@20 0.650  R@20 0.591  P@50 0.380
+ historical peak            AUPRC 0.855  P@20 0.750  R@20 0.682  P@50 0.440
full                         AUPRC 0.783  P@20 0.700  R@20 0.636  P@50 0.420


In [11]:
# how many critical sites are 'currently calm' (tamped) and only recoverable via peak memory?
c    = feats[feats.critical]
calm = c[c.current_amp < P["critical_mm"] * 0.8]        # sitting <24mm now
print(f"critical sites: {len(c)}   currently calm (<24mm now): {len(calm)}   "
      f"median historical max of the calm ones: {calm.max_amp.median():.1f}mm")

critical sites: 22   currently calm (<24mm now): 6   median historical max of the calm ones: 43.7mm


In [12]:
# lock the parsimonious model
for cols, name in [
    (["current_amp","max_amp"],            "current + peak (2)"),
    (["current_amp","max_amp","n_resets"], "current + peak + resets (3)"),
]:
    loo_scorecard(cols, name)

current + peak (2)           AUPRC 0.853  P@20 0.750  R@20 0.682  P@50 0.440
current + peak + resets (3)  AUPRC 0.851  P@20 0.750  R@20 0.682  P@50 0.440


In [13]:
# cell 7 — freeze exp02 model + honest scorecard, save artifacts
import json
from cprics import escalation as esc

labels = esc.site_labels(isl).reindex(feats["site_id"]).fillna(False)
cols = ["current_amp", "max_amp"]

score_by_site = esc.loo_escalation_scores(feats, labels.values, cols)
T = site_eval_table(isl, score_by_site, arch, P["concern_mm"], P["critical_mm"])
m = scorecard(T)

esc.save_escalation(esc.fit_escalation(feats, labels.values, cols),
                    config.MODELS_DIR / "exp02_supervised_9R_model.json")

mdir = config.OUTPUTS_DIR / "exp02_supervised_9R"; mdir.mkdir(parents=True, exist_ok=True)
with open(mdir / "metrics.json", "w") as fh:
    json.dump(m, fh, indent=2, default=str)

print(json.dumps(m, indent=2, default=str))
print("\nfrozen model:", esc.load_escalation(config.MODELS_DIR / "exp02_supervised_9R_model.json"))

{
  "n_sites": 2263,
  "n_critical": 22,
  "base_rate": 0.0097,
  "watchlist_size": 91,
  "flags_per_critical": 4.14,
  "precision": 0.242,
  "recall": 1.0,
  "auprc": 0.853,
  "precision_at_k": {
    "20": 0.75,
    "50": 0.44
  },
  "recall_at_k": {
    "20": 0.682,
    "50": 1.0
  },
  "advance_pct": 0.545,
  "median_lead_months": 12.0,
  "caught_ge_3mo": 9,
  "m11_advance_pct": 0.455,
  "m11_median_lead_months": 10.1,
  "fp_total": 69,
  "fp_by_archetype": {
    "Steady climber": 34,
    "Saw-tooth": 34,
    "Severe": 1
  }
}

frozen model: {'features': ['current_amp', 'max_amp'], 'mean': [2.506492353967651, 3.502841598297846], 'scale': [5.410957730750592, 6.832388687269497], 'coef': [1.225617009277009, 3.060102002204015], 'intercept': -13.158975409089425, 'n_train': 2263, 'n_positive': 22}
